# Stage-2 RHS optimization test

Compare three implementations:
1. `reference` = original code
2. `optimized_v1` = fast gamma lookup + matrix contractions
3. `optimized_v2` = v1 plus Q-level internal-cache reuse + vectorized bubble sums

In [1]:

import sys, time
from dataclasses import dataclass

import numpy as np
import pandas as pd

sys.path.insert(0, "/mnt/data")

import frg_kernel as k_ref
import frg_flow as f_ref
import frg_kernel_optimized as k_opt1
import frg_flow_optimized as f_opt1
import frg_kernel_optimized2 as k_opt2
import frg_flow_optimized2 as f_opt2

np.set_printoptions(precision=6, suppress=True)


## Build a tiny toy patch model

In [2]:

@dataclass
class ToyPatch:
    k_cart: np.ndarray
    energy: float

class ToyPatchSet:
    def __init__(self, ks, energies, b1=(2*np.pi, 0.0), b2=(0.0, 2*np.pi)):
        self.patches = [ToyPatch(np.asarray(k, dtype=float), float(e)) for k, e in zip(ks, energies)]
        self.Npatch = len(self.patches)
        self.b1 = np.asarray(b1, dtype=float)
        self.b2 = np.asarray(b2, dtype=float)

ks = [
    (0.20, 0.10),
    (1.10, 0.20),
    (0.30, 1.00),
    (1.20, 1.10),
]
energies_up = [-0.30, -0.10, 0.10, 0.35]
energies_dn = [-0.28, -0.08, 0.12, 0.33]

patchsets = {
    "up": ToyPatchSet(ks, energies_up),
    "dn": ToyPatchSet(ks, energies_dn),
}

def toy_bare_gamma(p1, s1, p2, s2, p3, s3, p4, s4):
    spin_factor = 1.0 if (s1 == s3 and s2 == s4) else 0.35
    patch_factor = (
        1.0
        + 0.03 * (p1 + 1)
        - 0.02 * (p2 + 1)
        + 0.04 * (p3 + 1)
        - 0.01 * (p4 + 1)
    )
    return spin_factor * patch_factor

pp_Qs = [np.array([0.0, 0.0]), np.array([1.3, 0.3])]
ph_Qs = [np.array([0.0, 0.0]), np.array([0.1, 0.9])]
phc_Qs = [np.array([0.0, 0.0]), np.array([0.9, 0.1])]


## Instantiate all three solvers

In [3]:

solver_ref = f_ref.FRGFlowSolver(
    patchsets=patchsets,
    bare_gamma=toy_bare_gamma,
    pp_Qs=pp_Qs,
    ph_Qs=ph_Qs,
    phc_Qs=phc_Qs,
    T_start=1.0,
    T_stop=0.7,
    n_steps=4,
    nfreq=12,
    diagnose_every=999,
)

solver_opt1 = f_opt1.FRGFlowSolver(
    patchsets=patchsets,
    bare_gamma=toy_bare_gamma,
    pp_Qs=pp_Qs,
    ph_Qs=ph_Qs,
    phc_Qs=phc_Qs,
    T_start=1.0,
    T_stop=0.7,
    n_steps=4,
    nfreq=12,
    diagnose_every=999,
)

solver_opt2 = f_opt2.FRGFlowSolver(
    patchsets=patchsets,
    bare_gamma=toy_bare_gamma,
    pp_Qs=pp_Qs,
    ph_Qs=ph_Qs,
    phc_Qs=phc_Qs,
    T_start=1.0,
    T_stop=0.7,
    n_steps=4,
    nfreq=12,
    diagnose_every=999,
)

print("spin blocks =", solver_ref.spin_blocks)
print("temperature path =", solver_ref.temperature_path)


spin blocks = [('up', 'up', 'up', 'up'), ('dn', 'dn', 'dn', 'dn'), ('up', 'dn', 'up', 'dn'), ('up', 'dn', 'dn', 'up'), ('dn', 'up', 'up', 'dn'), ('dn', 'up', 'dn', 'up')]
temperature path = [1.       0.887904 0.788374 0.7     ]


## 1. Scalar vertex-access consistency

In [4]:

gamma_ref = solver_ref.gamma_accessor()
gamma_opt1 = solver_opt1.current_gamma_accessor()
gamma_opt2 = solver_opt2.current_gamma_accessor()

test_points = [
    (0, "up", 1, "dn", 2, "up", 3, "dn"),
    (1, "up", 2, "up", 0, "up", 3, "up"),
    (2, "dn", 0, "up", 1, "dn", 3, "up"),
]

rows = []
for args in test_points:
    v_ref = gamma_ref(*args)
    v_opt1 = gamma_opt1(*args)
    v_opt2 = gamma_opt2(*args)
    rows.append({
        "args": str(args),
        "ref": v_ref,
        "opt1": v_opt1,
        "opt2": v_opt2,
        "abs_diff_ref_opt1": abs(v_ref - v_opt1),
        "abs_diff_ref_opt2": abs(v_ref - v_opt2),
        "abs_diff_opt1_opt2": abs(v_opt1 - v_opt2),
    })

pd.DataFrame(rows)


,args,ref,opt1,opt2,abs_diff_ref_opt1,abs_diff_ref_opt2,abs_diff_opt1_opt2
0,"(0, 'up', 1, 'dn', 2, 'up', 3, 'dn')",1.07+0.00j,1.07+0.00j,1.07+0.00j,0.0,0.0,0.0
1,"(1, 'up', 2, 'up', 0, 'up', 3, 'up')",1.00+0.00j,1.00+0.00j,1.00+0.00j,0.0,0.0,0.0
2,"(2, 'dn', 0, 'up', 1, 'dn', 3, 'up')",1.11+0.00j,1.11+0.00j,1.11+0.00j,0.0,0.0,0.0


## 2. Single-kernel consistency

In [5]:

cfg_ref = solver_ref._flow_config(float(solver_ref.temperature_path[0]))
cfg_opt1 = solver_opt1._flow_config(float(solver_opt1.temperature_path[0]))
cfg_opt2 = solver_opt2._flow_config(float(solver_opt2.temperature_path[0]))
Q0 = np.array([0.0, 0.0])

ker_pp_ref = k_ref.compute_pp_kernel(
    gamma_ref, patchsets, Q0,
    incoming_spins=("up", "dn"), outgoing_spins=("up", "dn"),
    config=cfg_ref, allowed_spin_blocks=solver_ref.allowed_spin_blocks,
)
ker_pp_opt1 = k_opt1.compute_pp_kernel_fast(
    gamma_opt1, patchsets, Q0,
    incoming_spins=("up", "dn"), outgoing_spins=("up", "dn"),
    config=cfg_opt1, allowed_spin_blocks=solver_opt1.allowed_spin_blocks,
)
ker_pp_opt2 = k_opt2.compute_pp_kernel_fast2(
    gamma_opt2, patchsets, Q0,
    incoming_spins=("up", "dn"), outgoing_spins=("up", "dn"),
    config=cfg_opt2, allowed_spin_blocks=solver_opt2.allowed_spin_blocks,
)

ker_ph_ref = k_ref.compute_ph_kernel(
    gamma_ref, patchsets, Q0,
    incoming_spins=("up", "up"), outgoing_spins=("up", "up"),
    config=cfg_ref, allowed_spin_blocks=solver_ref.allowed_spin_blocks,
)
ker_ph_opt1 = k_opt1.compute_ph_kernel_fast(
    gamma_opt1, patchsets, Q0,
    incoming_spins=("up", "up"), outgoing_spins=("up", "up"),
    config=cfg_opt1, allowed_spin_blocks=solver_opt1.allowed_spin_blocks,
)
ker_ph_opt2 = k_opt2.compute_ph_kernel_fast2(
    gamma_opt2, patchsets, Q0,
    incoming_spins=("up", "up"), outgoing_spins=("up", "up"),
    config=cfg_opt2, allowed_spin_blocks=solver_opt2.allowed_spin_blocks,
)

ker_phc_ref = k_ref.compute_phc_kernel(
    gamma_ref, patchsets, Q0,
    incoming_spins=("up", "dn"), outgoing_spins=("dn", "up"),
    config=cfg_ref, allowed_spin_blocks=solver_ref.allowed_spin_blocks,
)
ker_phc_opt1 = k_opt1.compute_phc_kernel_fast(
    gamma_opt1, patchsets, Q0,
    incoming_spins=("up", "dn"), outgoing_spins=("dn", "up"),
    config=cfg_opt1, allowed_spin_blocks=solver_opt1.allowed_spin_blocks,
)
ker_phc_opt2 = k_opt2.compute_phc_kernel_fast2(
    gamma_opt2, patchsets, Q0,
    incoming_spins=("up", "dn"), outgoing_spins=("dn", "up"),
    config=cfg_opt2, allowed_spin_blocks=solver_opt2.allowed_spin_blocks,
)

pd.DataFrame([
    {"kernel": "pp", "ref-opt1": np.max(np.abs(ker_pp_ref.matrix - ker_pp_opt1.matrix)), "ref-opt2": np.max(np.abs(ker_pp_ref.matrix - ker_pp_opt2.matrix)), "opt1-opt2": np.max(np.abs(ker_pp_opt1.matrix - ker_pp_opt2.matrix))},
    {"kernel": "ph", "ref-opt1": np.max(np.abs(ker_ph_ref.matrix - ker_ph_opt1.matrix)), "ref-opt2": np.max(np.abs(ker_ph_ref.matrix - ker_ph_opt2.matrix)), "opt1-opt2": np.max(np.abs(ker_ph_opt1.matrix - ker_ph_opt2.matrix))},
    {"kernel": "phc", "ref-opt1": np.max(np.abs(ker_phc_ref.matrix - ker_phc_opt1.matrix)), "ref-opt2": np.max(np.abs(ker_phc_ref.matrix - ker_phc_opt2.matrix)), "opt1-opt2": np.max(np.abs(ker_phc_opt1.matrix - ker_phc_opt2.matrix))},
])


,kernel,ref-opt1,ref-opt2,opt1-opt2
0,pp,4.440892e-16,4.441850e-16,4.441595e-16
1,ph,8.881784e-16,8.912022e-16,8.913677e-16
2,phc,8.881784e-16,8.912022e-16,8.913677e-16


## 3. Full RHS consistency + total timing

In [6]:

T0 = float(solver_ref.temperature_path[0])

def max_store_diff(a, b):
    keys = sorted(set(a.keys()) & set(b.keys()))
    vals = [np.max(np.abs(a[k] - b[k])) for k in keys]
    return float(max(vals)) if vals else 0.0

rhs_all = {}
time_all = {}
for name, solver in [("reference", solver_ref), ("optimized_v1", solver_opt1), ("optimized_v2", solver_opt2)]:
    t = time.perf_counter()
    rhs_all[name] = solver.compute_channel_rhs(T0)
    time_all[name] = time.perf_counter() - t

pd.DataFrame([
    {"store": "rhs_pp", "ref-opt1": max_store_diff(rhs_all["reference"][0], rhs_all["optimized_v1"][0]), "ref-opt2": max_store_diff(rhs_all["reference"][0], rhs_all["optimized_v2"][0]), "opt1-opt2": max_store_diff(rhs_all["optimized_v1"][0], rhs_all["optimized_v2"][0])},
    {"store": "rhs_phd", "ref-opt1": max_store_diff(rhs_all["reference"][1], rhs_all["optimized_v1"][1]), "ref-opt2": max_store_diff(rhs_all["reference"][1], rhs_all["optimized_v2"][1]), "opt1-opt2": max_store_diff(rhs_all["optimized_v1"][1], rhs_all["optimized_v2"][1])},
    {"store": "rhs_phc", "ref-opt1": max_store_diff(rhs_all["reference"][2], rhs_all["optimized_v1"][2]), "ref-opt2": max_store_diff(rhs_all["reference"][2], rhs_all["optimized_v2"][2]), "opt1-opt2": max_store_diff(rhs_all["optimized_v1"][2], rhs_all["optimized_v2"][2])},
    {"store": "time_reference", "ref-opt1": time_all["reference"], "ref-opt2": time_all["reference"], "opt1-opt2": time_all["reference"]},
    {"store": "time_optimized_v1", "ref-opt1": time_all["optimized_v1"], "ref-opt2": time_all["optimized_v1"], "opt1-opt2": time_all["optimized_v1"]},
    {"store": "time_optimized_v2", "ref-opt1": time_all["optimized_v2"], "ref-opt2": time_all["optimized_v2"], "opt1-opt2": time_all["optimized_v2"]},
    {"store": "speedup_ref_over_v1", "ref-opt1": time_all["reference"] / time_all["optimized_v1"], "ref-opt2": np.nan, "opt1-opt2": np.nan},
    {"store": "speedup_ref_over_v2", "ref-opt1": np.nan, "ref-opt2": time_all["reference"] / time_all["optimized_v2"], "opt1-opt2": np.nan},
    {"store": "speedup_v1_over_v2", "ref-opt1": np.nan, "ref-opt2": np.nan, "opt1-opt2": time_all["optimized_v1"] / time_all["optimized_v2"]},
])


,store,ref-opt1,ref-opt2,opt1-opt2
0,rhs_pp,6.661338e-16,8.881834e-16,8.881823e-16
1,rhs_phd,1.776357e-15,1.776703e-15,1.776703e-15
2,rhs_phc,1.776357e-15,1.776382e-15,8.913677e-16
3,time_reference,2.368979e+00,2.368979e+00,2.368979e+00
4,time_optimized_v1,3.731500e-01,3.731500e-01,3.731500e-01
5,time_optimized_v2,2.477270e-02,2.477270e-02,2.477270e-02
6,speedup_ref_over_v1,6.348596e+00,NaN,NaN
7,speedup_ref_over_v2,NaN,9.562860e+01,NaN
8,speedup_v1_over_v2,NaN,NaN,1.506295e+01


## 4. Manual per-channel timing (avoid timing one giant function)

In [7]:

def time_rhs_by_channel_ref(solver, T):
    gamma = solver.gamma_accessor()
    cfg = solver._flow_config(T)
    timings = {}

    t = time.perf_counter()
    rhs_pp = solver._empty_channel_store(solver.pp_grid)
    for iq, Q in enumerate(solver.pp_grid.q_list):
        for s1, s2, s3, s4 in solver.spin_blocks:
            ker = k_ref.compute_pp_kernel(gamma, solver.patchsets, Q, incoming_spins=(s1, s2), outgoing_spins=(s3, s4), config=cfg, allowed_spin_blocks=solver.allowed_spin_blocks)
            rhs_pp[(s1, s2, s3, s4, iq)] = np.asarray(ker.matrix, dtype=complex)
    timings["pp_time"] = time.perf_counter() - t

    t = time.perf_counter()
    rhs_phd = solver._empty_channel_store(solver.phd_grid)
    for iq, Q in enumerate(solver.phd_grid.q_list):
        for s1, s2, s3, s4 in solver.spin_blocks:
            ker = k_ref.compute_ph_kernel(gamma, solver.patchsets, Q, incoming_spins=(s1, s3), outgoing_spins=(s4, s2), config=cfg, allowed_spin_blocks=solver.allowed_spin_blocks)
            rhs_phd[(s1, s2, s3, s4, iq)] = np.asarray(ker.matrix, dtype=complex)
    timings["phd_time"] = time.perf_counter() - t

    t = time.perf_counter()
    rhs_phc = solver._empty_channel_store(solver.phc_grid)
    for iq, Q in enumerate(solver.phc_grid.q_list):
        for s1, s2, s3, s4 in solver.spin_blocks:
            ker = k_ref.compute_phc_kernel(gamma, solver.patchsets, Q, incoming_spins=(s1, s2), outgoing_spins=(s3, s4), config=cfg, allowed_spin_blocks=solver.allowed_spin_blocks)
            rhs_phc[(s1, s2, s3, s4, iq)] = np.asarray(ker.matrix, dtype=complex)
    timings["phc_time"] = time.perf_counter() - t
    timings["total_time"] = timings["pp_time"] + timings["phd_time"] + timings["phc_time"]
    return timings, (rhs_pp, rhs_phd, rhs_phc)


def time_rhs_by_channel_opt1(solver, T):
    gamma = solver.current_gamma_accessor()
    cfg = solver._flow_config(T)
    timings = {}

    t = time.perf_counter()
    rhs_pp = solver._empty_channel_store(solver.pp_grid)
    for iq, Q in enumerate(solver.pp_grid.q_list):
        for s1, s2, s3, s4 in solver.spin_blocks:
            ker = k_opt1.compute_pp_kernel_fast(gamma, solver.patchsets, Q, incoming_spins=(s1, s2), outgoing_spins=(s3, s4), config=cfg, allowed_spin_blocks=solver.allowed_spin_blocks)
            rhs_pp[(s1, s2, s3, s4, iq)] = np.asarray(ker.matrix, dtype=complex)
    timings["pp_time"] = time.perf_counter() - t

    t = time.perf_counter()
    rhs_phd = solver._empty_channel_store(solver.phd_grid)
    for iq, Q in enumerate(solver.phd_grid.q_list):
        for s1, s2, s3, s4 in solver.spin_blocks:
            ker = k_opt1.compute_ph_kernel_fast(gamma, solver.patchsets, Q, incoming_spins=(s1, s3), outgoing_spins=(s4, s2), config=cfg, allowed_spin_blocks=solver.allowed_spin_blocks)
            rhs_phd[(s1, s2, s3, s4, iq)] = np.asarray(ker.matrix, dtype=complex)
    timings["phd_time"] = time.perf_counter() - t

    t = time.perf_counter()
    rhs_phc = solver._empty_channel_store(solver.phc_grid)
    for iq, Q in enumerate(solver.phc_grid.q_list):
        for s1, s2, s3, s4 in solver.spin_blocks:
            ker = k_opt1.compute_phc_kernel_fast(gamma, solver.patchsets, Q, incoming_spins=(s1, s2), outgoing_spins=(s3, s4), config=cfg, allowed_spin_blocks=solver.allowed_spin_blocks)
            rhs_phc[(s1, s2, s3, s4, iq)] = np.asarray(ker.matrix, dtype=complex)
    timings["phc_time"] = time.perf_counter() - t
    timings["total_time"] = timings["pp_time"] + timings["phd_time"] + timings["phc_time"]
    return timings, (rhs_pp, rhs_phd, rhs_phc)


def time_rhs_by_channel_opt2(solver, T):
    gamma = solver.current_gamma_accessor()
    timings = {}

    t = time.perf_counter()
    cfg, pp_internal_by_iq, ph_internal_by_iq = solver._build_q_level_internal_caches(T)
    phc_internal_by_iq = {}
    for iq, Q in enumerate(solver.phc_grid.q_list):
        shift_cache = {(sa, sb): solver._phc_kplus[(iq, sb)] for sa, sb in k_opt2.available_internal_spin_pairs(solver.patchsets)}
        phc_internal_by_iq[iq] = k_opt2.build_ph_internal_cache_vec(solver.patchsets, Q, cfg, shift_cache=shift_cache)
    timings["cache_build_time"] = time.perf_counter() - t

    t = time.perf_counter()
    rhs_pp = solver._empty_channel_store(solver.pp_grid)
    for iq, Q in enumerate(solver.pp_grid.q_list):
        internal_cache = pp_internal_by_iq[iq]
        for s1, s2, s3, s4 in solver.spin_blocks:
            ker = k_opt2.compute_pp_kernel_fast2(gamma, solver.patchsets, Q, incoming_spins=(s1, s2), outgoing_spins=(s3, s4), config=cfg, allowed_spin_blocks=solver.allowed_spin_blocks, internal_cache=internal_cache, partner_in_resid=solver._pp_qminus[(iq, s2)], partner_out_resid=solver._pp_qminus[(iq, s4)])
            rhs_pp[(s1, s2, s3, s4, iq)] = np.asarray(ker.matrix, dtype=complex)
    timings["pp_time"] = time.perf_counter() - t

    t = time.perf_counter()
    rhs_phd = solver._empty_channel_store(solver.phd_grid)
    for iq, Q in enumerate(solver.phd_grid.q_list):
        internal_cache = ph_internal_by_iq[iq]
        for s1, s2, s3, s4 in solver.spin_blocks:
            ker = k_opt2.compute_ph_kernel_fast2(gamma, solver.patchsets, Q, incoming_spins=(s1, s3), outgoing_spins=(s4, s2), config=cfg, allowed_spin_blocks=solver.allowed_spin_blocks, internal_cache=internal_cache, partner_in_resid=solver._ph_kplus[(iq, s3)], partner_out_resid=solver._ph_kplus[(iq, s2)])
            rhs_phd[(s1, s2, s3, s4, iq)] = np.asarray(ker.matrix, dtype=complex)
    timings["phd_time"] = time.perf_counter() - t

    t = time.perf_counter()
    rhs_phc = solver._empty_channel_store(solver.phc_grid)
    for iq, Q in enumerate(solver.phc_grid.q_list):
        internal_cache = phc_internal_by_iq[iq]
        for s1, s2, s3, s4 in solver.spin_blocks:
            ker = k_opt2.compute_phc_kernel_fast2(gamma, solver.patchsets, Q, incoming_spins=(s1, s2), outgoing_spins=(s3, s4), config=cfg, allowed_spin_blocks=solver.allowed_spin_blocks, internal_cache=internal_cache, partner_in_resid=solver._phc_kminus[(iq, s4)], partner_out_resid=solver._phc_kplus[(iq, s3)])
            rhs_phc[(s1, s2, s3, s4, iq)] = np.asarray(ker.matrix, dtype=complex)
    timings["phc_time"] = time.perf_counter() - t
    timings["total_time"] = timings["cache_build_time"] + timings["pp_time"] + timings["phd_time"] + timings["phc_time"]
    return timings, (rhs_pp, rhs_phd, rhs_phc)


In [8]:

timing_ref, rhs_ref_manual = time_rhs_by_channel_ref(solver_ref, T0)
timing_opt1, rhs_opt1_manual = time_rhs_by_channel_opt1(solver_opt1, T0)
timing_opt2, rhs_opt2_manual = time_rhs_by_channel_opt2(solver_opt2, T0)

rows = [
    {"version": "reference", **timing_ref},
    {"version": "optimized_v1", **timing_opt1},
    {"version": "optimized_v2", **timing_opt2},
    {
        "version": "speedup ref/v1",
        "pp_time": timing_ref["pp_time"] / timing_opt1["pp_time"],
        "phd_time": timing_ref["phd_time"] / timing_opt1["phd_time"],
        "phc_time": timing_ref["phc_time"] / timing_opt1["phc_time"],
        "total_time": timing_ref["total_time"] / timing_opt1["total_time"],
    },
    {
        "version": "speedup ref/v2",
        "pp_time": timing_ref["pp_time"] / timing_opt2["pp_time"],
        "phd_time": timing_ref["phd_time"] / timing_opt2["phd_time"],
        "phc_time": timing_ref["phc_time"] / timing_opt2["phc_time"],
        "total_time": timing_ref["total_time"] / timing_opt2["total_time"],
    },
    {
        "version": "speedup v1/v2",
        "pp_time": timing_opt1["pp_time"] / timing_opt2["pp_time"],
        "phd_time": timing_opt1["phd_time"] / timing_opt2["phd_time"],
        "phc_time": timing_opt1["phc_time"] / timing_opt2["phc_time"],
        "total_time": timing_opt1["total_time"] / timing_opt2["total_time"],
    },
]

pd.DataFrame(rows)


,version,pp_time,phd_time,phc_time,total_time,cache_build_time
0,reference,0.530464,0.881518,0.907949,2.319930,NaN
1,optimized_v1,0.128719,0.142498,0.127391,0.398608,NaN
2,optimized_v2,0.004777,0.009289,0.009426,0.025130,0.001639
3,speedup ref/v1,4.121103,6.186174,7.127235,5.820074,NaN
4,speedup ref/v2,111.045405,94.901171,96.324860,92.316065,NaN
5,speedup v1/v2,26.945552,15.340851,13.515038,15.861665,NaN


## 5. Manual one-step timing breakdown

In [9]:

def manual_one_step(solver, T_old, dT):
    t0 = time.perf_counter()
    rhs_pp, rhs_phd, rhs_phc = solver.compute_channel_rhs(T_old)
    t_rhs = time.perf_counter() - t0

    t0 = time.perf_counter()
    effective_norm = max(solver.state.channel_norm(), solver.bare_vertex_norm, 1e-14)
    rhs_norm = solver._rhs_norm(rhs_pp, rhs_phd, rhs_phc)
    rel_update = abs(dT) * rhs_norm / effective_norm
    n_sub = max(1, int(np.ceil(rel_update / solver.max_relative_update))) if rel_update > solver.max_relative_update else 1
    sub_dT = dT / n_sub
    t_control = time.perf_counter() - t0

    t0 = time.perf_counter()
    for _ in range(n_sub):
        solver._apply_rhs(rhs_pp, rhs_phd, rhs_phc, sub_dT)
    solver.state.T = float(T_old + dT)
    t_apply = time.perf_counter() - t0

    return {
        "T_old": T_old,
        "dT": dT,
        "rhs_norm": rhs_norm,
        "effective_norm": effective_norm,
        "rel_update": rel_update,
        "n_sub": n_sub,
        "channel_norm_after": solver.state.channel_norm(),
        "t_rhs": t_rhs,
        "t_control": t_control,
        "t_apply": t_apply,
        "t_total": t_rhs + t_control + t_apply,
    }

solver_ref_step = f_ref.FRGFlowSolver(patchsets=patchsets, bare_gamma=toy_bare_gamma, pp_Qs=pp_Qs, ph_Qs=ph_Qs, phc_Qs=phc_Qs, T_start=1.0, T_stop=0.7, n_steps=4, nfreq=12, diagnose_every=999)
solver_opt1_step = f_opt1.FRGFlowSolver(patchsets=patchsets, bare_gamma=toy_bare_gamma, pp_Qs=pp_Qs, ph_Qs=ph_Qs, phc_Qs=phc_Qs, T_start=1.0, T_stop=0.7, n_steps=4, nfreq=12, diagnose_every=999)
solver_opt2_step = f_opt2.FRGFlowSolver(patchsets=patchsets, bare_gamma=toy_bare_gamma, pp_Qs=pp_Qs, ph_Qs=ph_Qs, phc_Qs=phc_Qs, T_start=1.0, T_stop=0.7, n_steps=4, nfreq=12, diagnose_every=999)

T_old = float(solver_ref_step.temperature_path[0])
dT = float(solver_ref_step.temperature_path[1] - solver_ref_step.temperature_path[0])

rec_ref = manual_one_step(solver_ref_step, T_old, dT)
rec_opt1 = manual_one_step(solver_opt1_step, T_old, dT)
rec_opt2 = manual_one_step(solver_opt2_step, T_old, dT)

pd.DataFrame([
    {"version": "reference", **rec_ref},
    {"version": "optimized_v1", **rec_opt1},
    {"version": "optimized_v2", **rec_opt2},
])


,version,T_old,dT,rhs_norm,effective_norm,rel_update,n_sub,channel_norm_after,t_rhs,t_control,t_apply,t_total
0,reference,1.0,-0.112096,5.195841,1.25,0.465946,4,0.582433,2.326032,0.000483,0.001336,2.327851
1,optimized_v1,1.0,-0.112096,5.195841,1.25,0.465946,4,0.582433,0.379230,0.000527,0.000377,0.380134
2,optimized_v2,1.0,-0.112096,5.195841,1.25,0.465946,4,0.582433,0.025595,0.000466,0.000373,0.026434


## 6. Manual multi-step loop (track each step explicitly)

In [10]:

def manual_flow_loop(solver, n_steps_to_run=3):
    rows = []
    temps = solver.temperature_path
    for i in range(min(n_steps_to_run, len(temps) - 1)):
        T_old = float(temps[i])
        dT = float(temps[i + 1] - temps[i])
        rec = manual_one_step(solver, T_old, dT)
        rec["step"] = i
        rows.append(rec)
    return pd.DataFrame(rows)

solver_ref_loop = f_ref.FRGFlowSolver(patchsets=patchsets, bare_gamma=toy_bare_gamma, pp_Qs=pp_Qs, ph_Qs=ph_Qs, phc_Qs=phc_Qs, T_start=1.0, T_stop=0.7, n_steps=5, nfreq=12, diagnose_every=999)
solver_opt1_loop = f_opt1.FRGFlowSolver(patchsets=patchsets, bare_gamma=toy_bare_gamma, pp_Qs=pp_Qs, ph_Qs=ph_Qs, phc_Qs=phc_Qs, T_start=1.0, T_stop=0.7, n_steps=5, nfreq=12, diagnose_every=999)
solver_opt2_loop = f_opt2.FRGFlowSolver(patchsets=patchsets, bare_gamma=toy_bare_gamma, pp_Qs=pp_Qs, ph_Qs=ph_Qs, phc_Qs=phc_Qs, T_start=1.0, T_stop=0.7, n_steps=5, nfreq=12, diagnose_every=999)

df_ref_loop = manual_flow_loop(solver_ref_loop, n_steps_to_run=3)
df_opt1_loop = manual_flow_loop(solver_opt1_loop, n_steps_to_run=3)
df_opt2_loop = manual_flow_loop(solver_opt2_loop, n_steps_to_run=3)

print("reference")
display(df_ref_loop)
print("optimized_v1")
display(df_opt1_loop)
print("optimized_v2")
display(df_opt2_loop)


reference


,T_old,dT,rhs_norm,effective_norm,rel_update,n_sub,channel_norm_after,t_rhs,t_control,t_apply,t_total,step
0,1.000000,-0.085309,5.195841,1.25,0.354601,3,0.443251,2.276957,0.000500,0.000342,2.277799,0
1,0.914691,-0.078031,10.216279,1.25,0.637751,5,1.236688,2.269771,0.000481,0.000507,2.270758,1
2,0.836660,-0.071374,24.902050,1.25,1.421896,10,3.014058,2.268996,0.000491,0.000908,2.270395,2


optimized_v1


,T_old,dT,rhs_norm,effective_norm,rel_update,n_sub,channel_norm_after,t_rhs,t_control,t_apply,t_total,step
0,1.000000,-0.085309,5.195841,1.25,0.354601,3,0.443251,0.384658,0.000466,0.000287,0.385411,0
1,0.914691,-0.078031,10.216279,1.25,0.637751,5,1.236688,0.376496,0.000473,0.000506,0.377474,1
2,0.836660,-0.071374,24.902050,1.25,1.421896,10,3.014058,0.434084,0.000487,0.000933,0.435504,2


optimized_v2


,T_old,dT,rhs_norm,effective_norm,rel_update,n_sub,channel_norm_after,t_rhs,t_control,t_apply,t_total,step
0,1.000000,-0.085309,5.195841,1.25,0.354601,3,0.443251,0.024521,0.000469,0.000286,0.025277,0
1,0.914691,-0.078031,10.216279,1.25,0.637751,5,1.236688,0.024495,0.000531,0.000463,0.025488,1
2,0.836660,-0.071374,24.902050,1.25,1.421896,10,3.014058,0.024418,0.000455,0.000904,0.025777,2
